In [0]:
from pyspark.sql.functions import col, when, min, max, date_format, to_date, lit


In [0]:
df = spark.read.table("nyctaxi.01_bronze.yello_trips_raw")

In [0]:
df.select(
    date_format(min(col('tpep_pickup_datetime')), 'dd-MM-yyyy').alias('oldest pickup date'),
    date_format(max(col('tpep_pickup_datetime')), 'dd-MM-yyyy').alias('latest pickup date')
).display()

In [0]:
df = df\
    .where(col('tpep_pickup_datetime') > to_date(lit('01-01-2026'), 'dd-MM-yyyy'))\
    .select(
        when(col('VendorID') == 1, 'Creative Mobile Technologies, LLC')
        .when(col('VendorID') == 6, 'Myle Technologies Inc')
        .when(col('VendorID') == 2, 'Curb Mobility, LLC')
        .otherwise('Helix')
        .alias('vendor'),

        col('tpep_pickup_datetime'), 

        col('tpep_dropoff_datetime'),

        col('passenger_count'),

        col('trip_distance'),

        when(col('RatecodeID') == 1 , 'Standard rate')
        .when(col('RatecodeID') == 2 , 'JFK')
        .when(col('RatecodeID') == 3 , 'Newark')
        .when(col('RatecodeID') == 4 , 'Nassau or Westchester')
        .when(col('RatecodeID') == 5 , 'Negotiated fare')
        .when(col('RatecodeID') == 6 , 'Group ride')
        .otherwise('Unknown')
        .alias('rate_type'),

        when(col('store_and_fwd_flag') == 'N', 'store and forward trip')
        .otherwise('not a store and forward trip')
        .alias('store_and_fwd_flag'),

        col('PULocationID').alias('pu_location_id'),

        col('DOLocationID').alias('do_location_id'),

        when(col('payment_type') == 0 , 'Flex Fare trip')
        .when(col('payment_type') == 1 , 'Credit card')
        .when(col('payment_type') == 2 , 'Cash')
        .when(col('payment_type') == 3 , 'No charge')
        .when(col('payment_type') == 4 , 'Dispute')
        .when(col('payment_type') == 6 , 'Voided trip')
        .otherwise('Unknown')
        .alias('payment_type'),

        col('fare_amount'),

        col('extra'),

        col('mta_tax'),

        col('tolls_amount'),

        col('improvement_surcharge'),

        col('total_amount'),

        col('congestion_surcharge'),

        col('Airport_fee').alias('airport_fee'), 

        col('cbd_congestion_fee')
    )

df.dtypes

In [0]:
df.write.mode("overwrite").saveAsTable("nyctaxi.02_silver.yellow_trips_cleansed")